# ISFC Analysis: Face − Shape

This notebook uses the Face − Shape contrast, keeps the three requested panels, and uses red for Face > Shape and blue for Shape > Face.

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import ttest_rel
from statsmodels.stats.multitest import multipletests

from matplotlib.patches import Wedge, PathPatch
from matplotlib.path import Path


# ============================================================
# SETTINGS
# ============================================================

n_rois = 90
alpha_fdr = 0.01

networks = ['TP', 'DMN', 'Limbic', 'FP', 'VIS', 'SUB']
n_networks = len(networks)

roi_labels = [
    str(i) if i % 10 == 0 else ''
    for i in range(1, n_rois + 1)
]


# ============================================================
# 1. ISFC - SHAPE
# ============================================================

data_shape = list(sub_timeseries_shape.values())

isfc_shape = np.asarray(
    isfc(data_shape, method='average')
)

mean_isfc_shape = isfc_shape.mean(axis=0)

print("Shape ISFC:", isfc_shape.shape)


# ============================================================
# 2. ISFC - FACE
# ============================================================

data_face = list(sub_timeseries_face.values())

isfc_face = np.asarray(
    isfc(data_face, method='average')
)

mean_isfc_face = isfc_face.mean(axis=0)

print("Face ISFC:", isfc_face.shape)


# ============================================================
# 3. PAIRED T-TEST
#
# t = Face - Shape
#
# t > 0  --> Face > Shape
# t < 0  --> Shape > Face
# ============================================================

t, p = ttest_rel(
    isfc_face,
    isfc_shape,
    axis=0
)


# ============================================================
# 4. FDR CORRECTION
#    ONLY UNIQUE ROI-TO-ROI EDGES
# ============================================================

triu_idx = np.triu_indices(
    n_rois,
    k=1
)

# 4005 unique edges
p_vec = p[triu_idx]

reject, p_fdr, _, _ = multipletests(
    p_vec,
    alpha=alpha_fdr,
    method='fdr_bh'
)

print("Number of tested unique edges:", len(p_vec))
print("Expected:", n_rois * (n_rois - 1) // 2)


# ============================================================
# 5. RECONSTRUCT SIGNIFICANCE MATRIX
# ============================================================

significant = np.zeros(
    (n_rois, n_rois),
    dtype=bool
)

significant[triu_idx] = reject

# Mirror upper triangle
significant = significant | significant.T

# Remove diagonal
np.fill_diagonal(
    significant,
    False
)


# ============================================================
# 6. SIGNIFICANT T-VALUE MATRIX
# ============================================================
sig_matrix = significant.astype(int)
#sig_matrix = np.zeros_like(t)

#sig_matrix[triu_idx] = np.where(
  #  reject,
  #  t[triu_idx],
  #  0
#)

# Mirror
sig_matrix = sig_matrix + sig_matrix.T

np.fill_diagonal(
    sig_matrix,
    0
)


# ============================================================
# 7. SEPARATE DIRECTIONS
#
# t = Face - Shape
#
# t > 0 --> Face > Shape
# t < 0 --> Shape > Face
# ============================================================

face_greater = (
    significant &
    (t > 0)
)

shape_greater = (
    significant &
    (t < 0)
)

np.fill_diagonal(
    face_greater,
    False
)

np.fill_diagonal(
    shape_greater,
    False
)


# ============================================================
# 8. SUMMARY OF SIGNIFICANT EDGES
# ============================================================

upper_sig = np.triu(
    significant,
    k=1
)

upper_face = np.triu(
    face_greater,
    k=1
)

upper_shape = np.triu(
    shape_greater,
    k=1
)

total_edges = n_rois * (n_rois - 1) // 2

n_significant = int(
    upper_sig.sum()
)

n_face_greater = int(
    upper_face.sum()
)

n_shape_greater = int(
    upper_shape.sum()
)

print()
print("========================================")
print("ISFC STATISTICAL SUMMARY")
print("========================================")
print("Total unique edges:", total_edges)
print("Significant edges:", n_significant)
print(
    "Percentage significant:",
    f"{100*n_significant/total_edges:.2f}%"
)
print("Face > Shape:", n_face_greater)
print("Shape > Face:", n_shape_greater)


# ============================================================
# 9. FUNCTION:
#    COUNT SIGNIFICANT EDGES BY NETWORK
# ============================================================

def network_edge_matrix(
    significance_matrix,
    networks,
    roi_network
):

    n = len(networks)

    matrix = np.zeros(
        (n, n),
        dtype=int
    )

    for i, net_i in enumerate(networks):

        rois_i = [
            roi
            for roi, net in roi_network.items()
            if net == net_i
        ]

        idx_i = np.asarray(
            rois_i,
            dtype=int
        ) - 1

        for j, net_j in enumerate(networks):

            rois_j = [
                roi
                for roi, net in roi_network.items()
                if net == net_j
            ]

            idx_j = np.asarray(
                rois_j,
                dtype=int
            ) - 1

            block = significance_matrix[
                np.ix_(idx_i, idx_j)
            ]

            # -----------------------------------------------
            # Within-network
            # Count only upper triangle
            # -----------------------------------------------

            if i == j:

                matrix[i, j] = int(
                    np.triu(
                        block,
                        k=1
                    ).sum()
                )

            # -----------------------------------------------
            # Between-network
            # -----------------------------------------------

            else:

                matrix[i, j] = int(
                    block.sum()
                )

    # Make matrix symmetric
    for i in range(n):

        for j in range(n):

            if i != j:

                matrix[j, i] = matrix[i, j]

    return matrix


# ============================================================
# 10. NETWORK MATRICES
# ============================================================

face_matrix_6net = network_edge_matrix(
    face_greater,
    networks,
    roi_network
)

shape_matrix_6net = network_edge_matrix(
    shape_greater,
    networks,
    roi_network
)

significant_matrix_6net = network_edge_matrix(
    significant,
    networks,
    roi_network
)


print()
print("Face > Shape network matrix:")
print(face_matrix_6net)

print()
print("Shape > Face network matrix:")
print(shape_matrix_6net)


# ============================================================
# 11. CHORD DIAGRAM FUNCTION
# ============================================================

def draw_chord(
    ax,
    matrix,
    labels,
    title,
    cmap='Blues'
):

    matrix = np.asarray(
        matrix,
        dtype=float
    ).copy()

    np.fill_diagonal(
        matrix,
        0
    )

    n = len(labels)

    # --------------------------------------------------------
    # Total connections per network
    # --------------------------------------------------------

    totals = matrix.sum(axis=1)

    total = totals.sum()

    # --------------------------------------------------------
    # Empty matrix
    # --------------------------------------------------------

    if total == 0:

        ax.text(
            0.5,
            0.5,
            'No significant connections',
            ha='center',
            va='center',
            fontsize=12,
            transform=ax.transAxes
        )

        ax.set_title(
            title,
            fontsize=13
        )

        ax.axis('off')

        return

    # --------------------------------------------------------
    # Angular positions
    # --------------------------------------------------------

    gap = 5

    available_angle = (
        360 -
        n * gap
    )

    angles = (
        available_angle *
        totals / total
    )

    start_angles = []
    end_angles = []

    current_angle = 90

    for angle in angles:

        start = current_angle

        end = (
            current_angle -
            angle
        )

        start_angles.append(start)
        end_angles.append(end)

        current_angle = (
            end - gap
        )

    # --------------------------------------------------------
    # Network arc
    # --------------------------------------------------------

    for i in range(n):

        wedge = Wedge(
            (0, 0),
            1.0,
            end_angles[i],
            start_angles[i],
            width=0.13,
            facecolor='white',
            edgecolor='black',
            linewidth=1.8
        )

        ax.add_patch(wedge)

        # Label
        theta = np.deg2rad(
            (start_angles[i] +
             end_angles[i]) / 2
        )

        r_label = 1.18

        x = (
            r_label *
            np.cos(theta)
        )

        y = (
            r_label *
            np.sin(theta)
        )

        rotation = np.rad2deg(theta)

        if x < 0:
            rotation += 180

        ax.text(
            x,
            y,
            labels[i],
            ha='center',
            va='center',
            fontsize=11,
            rotation=rotation
        )

    # --------------------------------------------------------
    # Maximum connection value
    # --------------------------------------------------------

    max_value = matrix.max()

    # --------------------------------------------------------
    # Draw chords
    # --------------------------------------------------------

    for i in range(n):

        for j in range(i + 1, n):

            value = matrix[i, j]

            if value <= 0:
                continue

            theta_i = np.deg2rad(
                (start_angles[i] +
                 end_angles[i]) / 2
            )

            theta_j = np.deg2rad(
                (start_angles[j] +
                 end_angles[j]) / 2
            )

            r = 0.87

            x1 = r * np.cos(theta_i)
            y1 = r * np.sin(theta_i)

            x2 = r * np.cos(theta_j)
            y2 = r * np.sin(theta_j)

            # Control points toward center
            cx1 = 0
            cy1 = 0

            cx2 = 0
            cy2 = 0

            verts = [
                (x1, y1),
                (cx1, cy1),
                (cx2, cy2),
                (x2, y2)
            ]

            codes = [
                Path.MOVETO,
                Path.CURVE4,
                Path.CURVE4,
                Path.CURVE4
            ]

            path = Path(
                verts,
                codes
            )

            # Chord thickness proportional
            # to number of significant edges
            linewidth = (
                1.5 +
                7.0 * value / max_value
            )

            patch = PathPatch(
                path,
                facecolor='none',
                edgecolor=plt.get_cmap(cmap)(
                    value / max_value
                ),
                linewidth=linewidth,
                alpha=0.75
            )

            ax.add_patch(patch)

    # --------------------------------------------------------
    # Formatting
    # --------------------------------------------------------

    ax.set_xlim(
        -1.35,
        1.35
    )

    ax.set_ylim(
        -1.35,
        1.35
    )

    ax.set_aspect(
        'equal'
    )

    ax.axis('off')

    ax.set_title(
        title,
        fontsize=13,
        pad=12
    )


# ============================================================
# 12. FINAL 3-PANEL FIGURE
#    Keep only panels 3, 4, and 5 from the original layout
#
#    3 = Significant ISFC differences: Face - Shape
#    4 = Face > Shape chord
#    5 = Shape > Face chord
# ============================================================

fig = plt.figure(figsize=(15, 5))

# ------------------------------------------------------------
# Panel 3. Significant ISFC differences: Face - Shape
# ------------------------------------------------------------

ax3 = plt.subplot(1, 3, 1)

sig_direction = np.zeros_like(t)

# Face > Shape: red (+1)
sig_direction[face_greater] = 1

# Shape > Face: blue (-1)
sig_direction[shape_greater] = -1

sns.heatmap(
    sig_direction,
    ax=ax3,
    cmap=['blue', 'white', 'red'],
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    xticklabels=roi_labels,
    yticklabels=roi_labels,
    cbar_kws={
        'label': 'Face − Shape',
        'ticks': [-1, 0, 1],
        'shrink': 0.6,
        'aspect': 30
    }
)

ax3.set_title(
    '3. Significant ISFC Differences: Face − Shape (FDR q < 0.01)',
    fontsize=13
)
ax3.set_xlabel('ROI')
ax3.set_ylabel('ROI')

# ------------------------------------------------------------
# Panel 4. Face > Shape chord
# ------------------------------------------------------------

ax4 = plt.subplot(1, 3, 2)

draw_chord(
    ax4,
    face_matrix_6net,
    networks,
    '4. Significant ISFC: Face > Shape',
    cmap='Reds'
)

# ------------------------------------------------------------
# Panel 5. Shape > Face chord
# ------------------------------------------------------------

ax5 = plt.subplot(1, 3, 3)

draw_chord(
    ax5,
    shape_matrix_6net,
    networks,
    '5. Significant ISFC: Shape > Face',
    cmap='Blues'
)

# ------------------------------------------------------------
# Overall layout
# ------------------------------------------------------------

plt.tight_layout(rect=[0, 0, 1, 0.94])

fig.suptitle(
    'Intersubject Functional Connectivity: Face − Shape',
    fontsize=16,
    y=0.995
)

plt.savefig(
    'ISFC_Final_3Panel_Chord_FaceMinusShape.png',
    dpi=600,
    bbox_inches='tight'
)

plt.show()
